# Homework 7
## Local Coupling (LoCo) between Land and Atmosphere

We will use data from the NASA MERRA-2 renalysis (daily output covering 1980-2020; 41 years) to investigate linear statistics of variables related to land-atmosphere interaction. First we will examine some univariate statistics, and then some bivariate statistics, concluding with the land-atmosphere coupling index.

* We will constrain our analysis to the region of the contiguous US (CONUS), which includes some of northern Mexico and southern Canada.
* We will further constrain ourselves to July - the peak month for land-atmosphere interactions over North America.
* The dataset you will use, `MERRA2_5stats_CONUS.nc4`, contains data relating to 16 variables that follow the diagram in Fig 12.1 from the class reading material. Specifics about the variables included are given below. The data in `MERRA2_5stats_CONUS.nc4` are the **five basic statistics** presented at the end of §12.2.2:
    * Means of each variable over 41 years of Julys: N = 1271 days
    * Variance across all days for each variable
    * Covariance across all days across all days between each pair of variables


In [ ]:
# Import useful software packages
import numpy as np
#import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.colors as colors

from cartopy import config          # cartopy is for plotting gridded spatial data and maps - developed by meteorologists at UKMO.
from cartopy import util as ut
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import (LongitudeFormatter, LongitudeLocator, LatitudeFormatter, LatitudeLocator)

import warnings
warnings.filterwarnings("ignore")

# Set some universal parameters
seconds_per_day = 86400
days_per_year = 365

### Coupling metrics

#### Functions
The set of functions below will allow calculation, for any variable:
* `lin5_std(dataset,var_a)` The standard deviation of `var_a` 
* `lin5_var(dataset,var_a)` The variance of `var_a` 
* `lin5_cv(dataset,var_a)` The coefficient of variation of `var_a` 

For any pair of variables, you can calculate:
* `lin5_cov(var_a,var_b)` The covariance between `var_a` and `var_b`
* `lin5_correl(var_a,var_b)` The Pearson's product moment correlation coefficient between `var_a` and `var_b`
* `lin5_slope(var_a,var_b)` The slope of the linear regression of `var_b` against `var_a`
* `lin5_ci(var_a,var_b)` The coupling index between forcing `var_a` and response `var_b`

For all these functions, `dataset` is the object that is the MERRA-2 xarray dataset, 
but `var_a` and `var_b` are <u>strings</u> of the names of the variables.

#### Plotting
The set of functions below will allow you to plot, for any variable:
* `plot_mean(var_a)` The mean of `var_a` 
* `plot_std(var_a)` The standard deviation of `var_a` 
* `plot_var(var_a)` The variance of `var_a` 
* `plot_cv(var_a)` The coefficient of variation of `var_a` 

For any pair of variables, you can plot:
* `plot_cov(var_a,var_b)` The covariance between `var_a` and `var_b`
* `plot_correl(var_a,var_b)` The Pearson's product moment correlation coefficient between `var_a` and `var_b`
* `plot_slope(var_a,var_b)` The slope of the linear regression of `var_b` against `var_a`
* `plot_ci(var_a,var_b)` The coupling index between forcing `var_a` and response `var_b`

As above, `var_a` and `var_b` are <u>strings</u> of the names of the variables, not the variable objects.

In [ ]:
def lin5_std(dataset,variable):
    """
    Calculates standard deviation of a variable from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable              (string) = Name of variable (really the name of its mean in the dataset)         
    Output:
        std_a       (xarray DataArray) = The population standard deviation of the variable, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable not in valid:
        raise KeyError("Requested variable not in Dataset.")
    var_sq = variable+"_"+variable                                        # Name of mean of squares variable
    std_a = np.sqrt(dataset[var_sq]-dataset[variable]*dataset[variable])  # Calculate 
    description = dataset[variable].attrs['long_name'].split("_of_")[1]   # Parse out long name of variable
    std_a.attrs['units'] = dataset[variable].attrs['units']               # Assign correct units attribute
    std_a.attrs['standard_name'] = "Standard_deviation_of_"+description   # Set `standard name`
    std_a.attrs['long_name'] = std_a.attrs['standard_name']               # Set `long name`
    return std_a

def lin5_var(dataset,variable):
    """
    Calculates standard deviation of a variable from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable              (string) = Name of variable (really the name of its mean in the dataset)         
    Output:
        var_a       (xarray DataArray) = The population variance of the variable, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable not in valid:
        raise KeyError("Requested variable not in Dataset.")
    var_sq = variable+"_"+variable                                        # Name of mean of squares variable
    var_a = dataset[var_sq]-dataset[variable]*dataset[variable]           # Calculate 
    description = dataset[variable].attrs['long_name'].split("_of_")[1]   # Parse out long name of variable
    var_a.attrs['units'] = "("+dataset[variable].attrs['units']+")**2"    # Assign correct units attribute
    var_a.attrs['standard_name'] = "Variance_of_"+description             # Set `standard name`
    var_a.attrs['long_name'] = var_a.attrs['standard_name']               # Set `long name`
    return var_a

def lin5_cv(dataset,variable):
    """
    Calculates coefficient of a variable from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable              (string) = Name of variable (really the name of its mean in the dataset)         
    Output:
        cv_a        (xarray DataArray) = The coefficient of variation of the variable, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable not in valid:
        raise KeyError("Requested variable not in Dataset.")
    var_sq = variable+"_"+variable                                        # Name of mean of squares variable
    cv_a = np.sqrt(dataset[var_sq]-dataset[variable]*dataset[variable]) \
                              / dataset[variable]                         # Calculate 
    description = dataset[variable].attrs['long_name'].split("_of_")[1]   # Parse out long name of variable
    cv_a.attrs['units'] = "1"                                             # Assign correct units attribute
    cv_a.attrs['standard_name'] = "Coefficient_of_variation_of_"+description   # Set `standard name`
    cv_a.attrs['long_name'] = cv_a.attrs['standard_name']                 # Set `long name`
    return cv_a

def lin5_cov(dataset,variable_a,variable_b):
    """
    Calculates covariance between two variables from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable_a            (string) = Name of first variable (really the name of its mean in the dataset)         
        variable_b            (string) = Name of second variable (really the name of its mean in the dataset)         
    Output:
        cov_ab      (xarray DataArray) = The population covariance between the variables, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable_a not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if variable_b not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if valid.index(variable_a) > valid.index(variable_b):                 # Swap variables if necessary
        variable_a,variable_b = variable_b,variable_a
    variable_ab = variable_a+"_"+variable_b                               # Name of product of variables
    cov_ab = dataset[variable_ab]-dataset[variable_a]*dataset[variable_b] # Calculate 
    desc_a = dataset[variable_a].attrs['long_name'].split("_of_")[1]      # Parse out long name of variable
    desc_b = dataset[variable_b].attrs['long_name'].split("_of_")[1]      # Parse out long name of variable
    cov_ab.attrs['units'] = dataset[variable_ab].attrs['units']           # Assign correct units attribute
    cov_ab.attrs['standard_name'] = "Covariance_between_"+desc_a+"_and_"+desc_b  # Set `standard name`
    cov_ab.attrs['long_name'] = cov_ab.attrs['standard_name']             # Set `long name`
    return cov_ab

def lin5_correl(dataset,variable_a,variable_b):
    """
    Calculates the Pearson's correlation coefficient between two variables from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable_a            (string) = Name of first variable (really the name of its mean in the dataset)         
        variable_b            (string) = Name of second variable (really the name of its mean in the dataset)         
    Output:
        corr_ab     (xarray DataArray) = The Pearson's correlation coefficient between the variables, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable_a not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if variable_b not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if valid.index(variable_a) > valid.index(variable_b):                 # Swap variables if necessary
        variable_a,variable_b = variable_b,variable_a
    var_a_sq = variable_a+"_"+variable_a                                  # Name of mean of squares of variable_a
    var_b_sq = variable_b+"_"+variable_b                                  # Name of mean of squares of variable_b
    variable_ab = variable_a+"_"+variable_b                               # Name of product of variables
    corr_ab = (dataset[variable_ab]-dataset[variable_a]*dataset[variable_b]) \
                / np.sqrt((dataset[var_a_sq]-dataset[variable_a]*dataset[variable_a]) \
                        * (dataset[var_b_sq]-dataset[variable_b]*dataset[variable_b]))          # Calculate 
    desc_a = dataset[variable_a].attrs['long_name'].split("_of_")[1]      # Parse out long name of variable
    desc_b = dataset[variable_b].attrs['long_name'].split("_of_")[1]      # Parse out long name of variable
    corr_ab.attrs['units'] = "1"                                          # Assign correct units attribute
    corr_ab.attrs['standard_name'] = "Correlation_between_"+desc_a+"_and_"+desc_b  # Set `standard name`
    corr_ab.attrs['long_name'] = corr_ab.attrs['standard_name']           # Set `long name`
    return corr_ab


def lin5_slope(dataset,variable_a,variable_b):
    """
    Calculates the slope of the best-fit linear regression of variable_b on variable_a from dataset of 5 linear statistics
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable_a            (string) = Name of independent variable (really the name of its mean in the dataset)         
        variable_b            (string) = Name of dependent variable (really the name of its mean in the dataset)         
    Output:
        sens_ab     (xarray DataArray) = The slope of the best-fit linear regression of variable_b on variable_a, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable_a not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if variable_b not in valid:
        raise KeyError("Requested variable not in Dataset.")
    var_ind = variable_a ; var_dep = variable_b                           # Keep straight the independent and dependent variables
    if valid.index(variable_a) > valid.index(variable_b):                 # Swap variables if necessary
        variable_a,variable_b = variable_b,variable_a
    var_ind_sq = var_ind+"_"+var_ind                                      # Name of mean of squares of independent variable
    variable_ab = variable_a+"_"+variable_b                               # Name of product of variables
    sens_ab = (dataset[variable_ab]-dataset[variable_a]*dataset[variable_b]) \
                / (dataset[var_ind_sq]-dataset[var_ind]*dataset[var_ind]) # Calculate 
    desc_ind = dataset[var_ind].attrs['long_name'].split("_of_")[1]       # Parse out long name of independent variable
    desc_dep = dataset[var_dep].attrs['long_name'].split("_of_")[1]       # Parse out long name of dependent variable
    sens_ab.attrs['units'] = "("+dataset[var_dep].attrs['units']+") / ("+dataset[var_ind].attrs['units']+")" # Assign correct units attribute
    sens_ab.attrs['standard_name'] = "Slope_of_"+desc_dep+"_regressed_on_"+desc_ind  # Set `standard name`
    sens_ab.attrs['long_name'] = sens_ab.attrs['standard_name']           # Set `long name`
    return sens_ab


def lin5_coupling_index(dataset,variable_a,variable_b):
    """
    Calculates coupling index between source variable_a on target variable_b from dataset of 5 linear statistics
    References:
    • Guo, et al., 2006: GLACE: The Global Land-Atmosphere Coupling Experiment. 2. Analysis. J. Hydrometeor., 7, 611-625, https://doi.org/10.1175/JHM511.1 
    • Dirmeyer, 2011: The terrestrial segment of soil moisture-climate coupling. Geophys. Res. Lett., 38, L16702, https://doi.org/10.1029/2011GL048268
    • Dirmeyer, et al., 2014: Intensified land surface control on boundary layer growth in a changing climate. Geophys. Res. Lett., 41, https://doi.org/10.1002/2013GL058826
    Required inputs:
        dataset       (xarray Dataset) = Dataset containing linear5 statistics
        variable_a            (string) = Name of forcing or source variable (really the name of its mean in the dataset)         
        variable_b            (string) = Name of responding or target variable (really the name of its mean in the dataset)         
    Output:
        ci_ab       (xarray DataArray) = The coupling index between source variable_a on target variable_b, with correct attributes
    """
    valid = [s for s in list(dataset.keys()) if "_" not in s]             # List of available variables
    if variable_a not in valid:
        raise KeyError("Requested variable not in Dataset.")
    if variable_b not in valid:
        raise KeyError("Requested variable not in Dataset.")
    var_sou = variable_a ; var_tar = variable_b                           # Keep straight the source and target variables
    var_sou_sq = var_sou+"_"+var_sou                                      # Name of mean of squares of source variable
    if valid.index(variable_a) > valid.index(variable_b):                 # Swap variables if necessary
        variable_a,variable_b = variable_b,variable_a
    variable_ab = variable_a+"_"+variable_b                               # Name of product of variables
    ci_ab = (dataset[variable_ab]-dataset[variable_a]*dataset[variable_b]) \
                / np.sqrt(dataset[var_sou_sq]-dataset[var_sou]*dataset[var_sou]) # Calculate 
    desc_sou = dataset[var_sou].attrs['long_name'].split("_of_")[1]       # Parse out long name of independent variable
    desc_tar = dataset[var_tar].attrs['long_name'].split("_of_")[1]       # Parse out long name of dependent variable
    ci_ab.attrs['units'] = dataset[var_tar].attrs['units']                # Assign correct units attribute
    ci_ab.attrs['standard_name'] = "Coupling_index_from_"+desc_sou+"_to_"+desc_tar  # Set `standard name`
    ci_ab.attrs['long_name'] = ci_ab.attrs['standard_name']               # Set `long name`
    return ci_ab

def plot_mean(variable):
    metric = ds[variable]
    plot_one(metric,r"$\mathbf{\overline{"+variable+"}}$")
    return

def plot_std(variable):
    metric = lin5_std(ds,variable)
    plot_one(metric,r"$\mathbf{\sigma_{"+variable+"}}$")
    return

def plot_var(variable):
    metric = lin5_var(ds,variable)
    plot_one(metric,r"$\mathbf{\sigma_{"+variable+"}^{2}}$")
    return

def plot_cv(variable):
    metric = lin5_cv(ds,variable)
    plot_one(metric,r"$\mathbf{\sigma_{"+variable+"}/\overline{"+variable+"}}$")
    return

def plot_cov(variable_a,variable_b):
    metric = lin5_cov(ds,variable_a,variable_b)
    plot_two(metric,f"cov({variable_a},{variable_b})")
    return

def plot_correl(variable_a,variable_b):
    metric = lin5_correl(ds,variable_a,variable_b)
    plot_two(metric,f"r({variable_a},{variable_b})")
    return

def plot_slope(variable_a,variable_b):
    metric = lin5_slope(ds,variable_a,variable_b)
    plot_two(metric,f"∆{variable_b}/∆{variable_a}")
    return

def plot_ci(variable_a,variable_b):
    metric = lin5_coupling_index(ds,variable_a,variable_b)
    plot_two(metric,f"r({variable_a},{variable_b})\n       $\cdot \sigma$({variable_b})")
    return

def plot_one(metric,fn):
    fig, ax = plt.subplots(figsize=(8,4.4),subplot_kw=dict(projection=ccrs.EckertIV(central_longitude=-95.0)))
    #ax = plt.subplot(projection=ccrs.EckertIV(central_longitude=-95.0))
    conus = ax.contourf(metric.lon,metric.lat,metric,20,cmap='plasma',transform=ccrs.PlateCarree()) 
    ax.set_xticks([])
    ax.set_yticks([])
    ax.coastlines()
    ax.add_feature(cfeature.STATES)
    ax.set_title(metric.attrs['long_name'].replace("_"," "),fontsize=14)
    ax.text(287.3,30.5,fn,ha='center',fontsize=16,fontweight='bold',transform=ccrs.PlateCarree())
    cbax0 = fig.add_axes([0.123, 0.09, 0.781, 0.03]) 
    barlab = ""
    if metric.attrs['units'] != "1":
        barlab = f"[{metric.attrs['units']}]"
    cbar0 = fig.colorbar(conus,cax=cbax0,orientation='horizontal')
    cbar0.ax.tick_params(labelsize=12)
    cbar0.set_label(label=barlab,size=14)
    return

def plot_two(metric,fn):
    fig, ax = plt.subplots(figsize=(8,4.4),subplot_kw=dict(projection=ccrs.EckertIV(central_longitude=-95.0)))
    #ax = plt.subplot(projection=ccrs.EckertIV(central_longitude=-95.0))
    conus = ax.contourf(metric.lon,metric.lat,metric,20,cmap='RdBu_r',
                        norm=colors.CenteredNorm(),transform=ccrs.PlateCarree()) 
    ax.coastlines()
    ax.add_feature(cfeature.STATES)
    ax.set_title(metric.attrs['long_name'].replace("_"," "),fontsize=14)
    ax.text(287.3,30.5,fn,ha='center',fontsize=16,fontweight='bold',transform=ccrs.PlateCarree())
    cbax0 = fig.add_axes([0.123, 0.09, 0.781, 0.03]) 
    barlab = ""
    if metric.attrs['units'] != "1":
        barlab = f"[{metric.attrs['units']}]"
    cbar0 = fig.colorbar(conus,cax=cbax0,orientation='horizontal')
    cbar0.ax.tick_params(labelsize=12)
    cbar0.set_label(label=barlab,size=14)
    return

def plot_E3():
    metric_a = lin5_coupling_index(ds,'sms','lhf')
    metric_b = lin5_coupling_index(ds,'sms','shf')
    metric = -metric_a-metric_b

    fig, ax = plt.subplots(figsize=(8,4.4),subplot_kw=dict(projection=ccrs.EckertIV(central_longitude=-95.0)))
    #ax = plt.subplot(projection=ccrs.EckertIV(central_longitude=-95.0))
    conus = ax.contourf(metric.lon,metric.lat,metric,20,cmap='RdBu_r',
                        norm=colors.CenteredNorm(),transform=ccrs.PlateCarree()) 
    ax.coastlines()
    ax.add_feature(cfeature.STATES)
    ax.set_title("Soil moisture couples relatively weakly to LHF (red) or SHF (blue)",fontsize=14)
    ax.text(287.3,29.0,"CI(sms,lhf)\nvs\nCI(sms,shf)",
            ha='center',fontsize=16,fontweight='bold',transform=ccrs.PlateCarree())
    cbax0 = fig.add_axes([0.123, 0.09, 0.781, 0.03]) 
    barlab = f"[{metric_a.attrs['units']}]"
    cbar0 = fig.colorbar(conus,cax=cbax0,orientation='horizontal')
    cbar0.ax.tick_params(labelsize=12)
    cbar0.set_label(label=barlab,size=14)
    return


### Read the Data

In the cell below, the dataset is read in, and then a table is plotted to show you the avaiable variables, their names and units. We can group these variables similarly to Figure 12.1:

|                 || Variables    |
|-----------------||--------------|
| Sources:        || Energy: `swd`, `lwd`; Moisture: `prec` |
| Land surface    || States: `sms`, `tss`; Available energy: `rnet` |
| Surface fluxes  || `lhf`, `shf`, `ef` |
| Near-surface atmospheric states || `t2m`, `q2m`, `wind` |
| Boundary layer properties || `lcl`, `me`, `pbl` |
| Clouds || `pdr` |

A few things to note:
* `prec` has units that give very small numbers, equivalent to rainfall rates in [*mm*/*s*] - multiply by 86400 to get [*mm*/*day*].
* `sms` is volumetric soil water content in the surface layer of the soil. The MERRA-2 name is rather cryptic. `tss` is temperature of the same soil layer.
* `pbl` is the daily mean height of the top of the PBL in pressure units, so it is small when the PBL is deep, and vice versa. 
* Moist enthalpy `me` is interchangeable with moist static energy in this case without any loss of meaning.
* `lcl` and `me` were calculated from 2m temperature and humidity. `lcl` is height above the ground [*m*], so it behaves as you might expect.
* The variable we use to indicate cloud is `pdr`, direct beam sunlight at the surface; `pdr=0` on overcast days, and it will be large when skies are clear - so it is inversely proportional to cloudiness.

In [ ]:
### Open data file
ds = xr.open_dataset("MERRA2_5stats_CONUS.nc4")
variables = [s for s in list(ds.keys()) if "_" not in s]   # These are the core variables, which each have means and means of squares
descriptions = [ds[v].attrs['long_name'].split("_of_")[1] for v in variables]  # List of descriptions of core variables
units = [ds[v].attrs['units'] for v in variables]                              # List of units of core variables

### Print out the list of variables
print(f"    Name  {'Description of variable':35}   Units")
print("-"*59)
for i in range(len(variables)):
    print(f"{i:2}  {variables[i]:>4}  {descriptions[i]:35}  [{units[i]}]")

----------------

## Part A. July Means:

Familiarize yourself with the summertime climatological means of the 16 variables by plotting each below.

In [ ]:
for v in variables:
    plot_mean(v) # This is a function we defined above to make nice standardized maps

### A: Answer a question

For each variable, give a description of the spatial pattern you see (the major features; where is it high, where is it low) and <u>why</u> the pattern is so. Give your reasons in terms of climate and, for the *derived* variables that are combinations of other variables (`rnet`,`ef`, `lcl`, `me`), how the combinations result in what you see.

-----------------

## Part B. Standard deviations:

Repeat the process from Part A below to plot and comment on the standard deviations of each variable. Remember, `plot_std` is showing the *day-to-day* variability of the variable - where there is changeable *weather* with regard to that variable, and where every day is similar to every other.

In [ ]:
for v in variables:
    plot_std(v)

### B: Answer a question

As in Part A, give a description of the patterns for each variable and how they likely arise.

---------------------

## Part C. Correlations:

Use Fig 12.1 as a guide - you are <u>not</u> expected to look at all 240 combinations of variables! 😁

The cell below is set up for you to address the first question - the loop will step through pairs of variables `var_a` and `var_b` and make plots for each pair.

In [ ]:
var_a = ['sms','rnet','prec']
var_b = ['lhf','lhf','lhf']

for i in range(len(var_a)):
    plot_correl(var_a[i],var_b[i])

### C: Answer a bunch of questions

1. The cell is set up to plot 3 correlation maps: r(sms,lhf), r(rnet,lhf), r(prec,lhf). The first two relate to the idea of "energy limit" versus "water limit" on evaporation. Explain what regions are in each category and why the maps tell you this. Then compare to the last map and hypothesize the relationship between precipitation and soil moisture.
2. Make new lists for `var_a` and `var_b` so you may plot the correlations of soil temperature to sensible heat flux, and soil moisture to sensible heat flux. What process links soil temperature and senible heat flux when r(tss,shf)>>0? When r(tss,shf)<<0? When you compare that map to the map for r(sms,shf), what do you see? Predict what a map of r(sms,tss) would look like, then plot it (honor system not to peek before answering). Was your prediction good? Why?
3. Moving to the next level - surface fluxes and 2m air states. Set up `var_a` and `var_b` so you plot correlations of sensible heat flux with air temperature, and latent heat flux with specific humidity. It makes sense that each flux, when large, should lead to an increase in its companion atmospheric state variable, which would result in positive correlations (red). But each map has at least a couple regions with negative correlations (blue). What process(es) are different there - what's going on? (the specific regions may give you some clues what is happening)
4. Correlate each of the two surface fluxes with the wind speed. Describe what is happening where r(shf,wind)>>0 (recall the terms in the equation for heat fluxes - which one is a function of wind speed?). Then describe what is happening where r(lhf,wind)<<0 (again, recall our discussions about evapotranspiration).
5. Correlate 2m air temperature with each of moist enthalpy (ME), LCL, and PBL top pressure. Recall how $T_{2m}$ plays into the formulations for ME and LCL. What must be happening in the regions where r(t2m,me) are small or negative? Likewise for r(t2m,lcl), what is happening around the Great Lakes and New England? Compare r(t2m,lcl) to r(t2m,pbl) recalling the definition of `pbl` such that small values (pressures) mean deep boundary layers. How well do they agree, and what can you infer about LCL as an approximation for PBL height?  You can have a look at r(lcl,pbl) to see where they do/don't agree.
6. Repeat 5. but for `q2m` instead of `t2m`. What can you say about humidity's contribution to these terms? Where is humidity more important than temperature as a control on PBL characteristics?
7. Now at the top of our schematic, let's look at cloud formation. Correlate ME, LCL, and PBL top pressure each with `pdr` (remember low `pdr` indcates clouds, high `pdr` is lack of cloud). Keeping the signs straight, where does ME appear to be an important factor for cloud formation? What about LCL and/or PBL? Where are they both important, where is one clearly more important (and why do you think this is so)? 
8. Finally, let's look at the direct link between precipitation and soil moisture. You might expect them to be highly correlated everywhere, and the correlation is positive nearly everywhere (California, you may have noticed by now, is often strange). But there is a big north-south stripe of strong positive correlations - where is it, why, and what is such a region called?

-------------

## Part D. Slope/Sensitivity:

Part C was pretty intense. let's look at just a couple of sensitivity relationships.

The cell below is set up for you to address the first question.

In [ ]:
var_a = ['sms','rnet']
var_b = ['lhf','lhf']

for i in range(len(var_a)):
    plot_slope(var_a[i],var_b[i])

### D: Answer a couple questions

1. The cell is set up to plot 2 sensitivity maps: ∆lhf/∆sms and ∆lhf/∆rnet (remember here, the order of the variables matters!). Where is there the greatest sensitivity of evaporation to fluctuations in soil moisture and how does this area relate, process-wise, to what was discussed above in part C.8? Where is the region of greatest sensitivity of evapotranspiration to net radiation and what is the dominant land use in that region? Describe how vegetation may be relevant to what you see.
2. Plot the sensitivity of precipitation to variations in soil moisture and compare that to what you found in part C.8. The Southeast US had a high correlation, but it has faded away in the sensitivity plot, while the Great Plains remains a dominant feature. Discuss the implications of high sensitivity of precipitation to changes in soil moisture.

---------------

## Part E. Coupling Indices:

Review the definition of coupling index (Equation 12.26) - both ways it can be expressed in terms of the statistics we ahve been looking at here. The cell below is set up for you to address the first question.

In [ ]:
var_a = ['rnet','rnet']
var_b = ['shf','lhf']

for i in range(len(var_a)):
    plot_ci(var_a[i],var_b[i])

### E: Answer some questions

1. The cell is set up for the coupling indices of net radiation as the source variable, latent heat flux and sensible heat flux as the target vairables. The maps show similar but opposite patterns. Where do variations in net radiation couple strongly to determine latent heat flux? They are in very different locations; how/why does each *make sense* based on what you've learned? There are two main "hot spot" regions where sensible heat flux appears strongly controlled by net radiation: the Southeast and the Southwest/Western Mexico. What should be the important processes in each region?
2. Plot maps of the coupling indices for soil moisture driving each of the two surface heat fluxes. Compared to the sensitivity map from part D for the soil moisture linkage to latent heat flux, the "hot spot"  has shifted to the southern Great Plains. Why? (clues: Eq. 12.26 and the relevant plot in Part B)
3. There is a code cell at the bottom of this notebook with a single line, which will produce a comparison of the coupling indices from the previous part, so you can more clearly see how they differ. There are two main regions where the coupling for soil moisture to surface latent heat flux is quite a bit weaker than for sensible heat flux (red areas). What different balances of processes in these two regions, different situations, would lead to these similar results? Don't forget the formulation of the coupling index as you construct your answer.

In [ ]:
# To answer E.3:
plot_E3()